# CatBoost v2: временная динамика и регулярность поведения

Этот ноутбук полностью описывает второй вариант решения и воспроизводит сегодняшний эксперимент:

1. строит расширенные пользовательские срезы `enhanced_v2`;
2. проверяет их на четырёх временных holdout;
3. сравнивает v2 с первым CatBoost baseline;
4. проверяет посткалибровку на OOF-прогнозах;
5. обучает финальную модель и создаёт `submission_catboost_v2.csv`.

Код расчёта признаков и обучения находится в `src/`. Ноутбук задаёт последовательность эксперимента, показывает параметры и сохраняет результаты.

## 0. Режим запуска

Тяжёлые стадии управляются флагами. По умолчанию используются уже рассчитанные срезы и метрики, поэтому ноутбук можно быстро открыть и изучить.

Для полного повторения эксперимента установите `RUN_MULTIFOLD_VALIDATION = True` и `TRAIN_FINAL_MODEL = True`. Для принудительного пересчёта Parquet-срезов установите также `REBUILD_SNAPSHOTS = True`.

In [ ]:
REBUILD_SNAPSHOTS = False
RUN_MULTIFOLD_VALIDATION = False
TRAIN_FINAL_MODEL = False

FEATURE_PACK = 'enhanced_v2'
FINAL_ITERATIONS = 300

## 1. Импорты и пути

Ноутбук можно запускать как из корня проекта, так и из папки `notebooks/`. Все пути определяются централизованно в `src/config.py`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_V2_ARTIFACT_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    CATBOOST_V2_SUBMISSION_PATH,
    MULTIFOLD_VALIDATION_DIR,
    VALIDATION_ANCHORS,
    ensure_output_dirs,
)
from src.data import all_users, latest_labeled_anchor, load_train, summarize_data, validate_user_days
from src.evaluation import save_json
from src.experiments import fit_final_model, run_temporal_experiment
from src.features import build_snapshot, load_snapshots, save_snapshot
from src.models import make_final_model, make_validation_model
from src.validation import feature_columns, make_historical_anchors, make_temporal_folds, rmsle

ensure_output_dirs()
print(f'Артефакты v2: {CATBOOST_V2_ARTIFACT_DIR}')
print(f'Будущий submission: {CATBOOST_V2_SUBMISSION_PATH}')

## 2. Что изменено относительно baseline

Baseline использует накопительные окна 7, 30 и 90 дней. Они хорошо описывают общий объём активности, но хуже показывают, в какой части истории она происходила.

В `enhanced_v2` добавлены:

- непересекающиеся периоды 8–30, 31–60, 61–90 и 91–180 дней;
- суммы GMV, заказов, корзин, поисков и канальных действий в этих периодах;
- число активных дней и среднее только по положительным дням;
- плотность наблюдаемой активности;
- стаж пользователя и давность любой активности;
- локальные тренды между соседними периодами;
- конверсии и доли каналов отдельно для 7, 30 и 90 дней.

Идентификатор `user_id` по-прежнему не передаётся модели. Цель остаётся прежней: `log1p` будущего 30-дневного GMV.

## 3. Загрузка исходных данных

Исходная таблица нужна для контроля дат и для построения срезов, если они отсутствуют или запрошен принудительный пересчёт.

In [ ]:
data = load_train()
validate_user_days(data)
summary = summarize_data(data)
users = all_users(data)
latest_anchor = latest_labeled_anchor(data)
historical_anchors = make_historical_anchors(latest_anchor)
final_anchor = summary['max_date']

print(summary)
print('Исторические якоря:', historical_anchors)
print('Финальный якорь:', final_anchor)

## 4. Построение или повторное использование v2-срезов

Каждый обучающий срез содержит 250 тысяч пользователей, признаки на дату якоря и известный GMV следующих 30 дней. Финальный срез содержит только признаки на 13 февраля 2026 года.

Если файл уже существует и `REBUILD_SNAPSHOTS=False`, он используется повторно. Это экономит время и гарантирует, что разные модели сравниваются на одной матрице признаков.

In [ ]:
for anchor in historical_anchors:
    path = CATBOOST_V2_SNAPSHOT_DIR / f'train_{anchor.isoformat()}.parquet'
    if REBUILD_SNAPSHOTS or not path.exists():
        print(f'Строим обучающий v2-срез {anchor} ...')
        snapshot = build_snapshot(
            data, users, anchor, with_target=True, feature_pack=FEATURE_PACK
        )
        save_snapshot(snapshot, CATBOOST_V2_SNAPSHOT_DIR, anchor, 'train')
    else:
        print(f'Используем готовый срез: {path.name}')

test_path = CATBOOST_V2_SNAPSHOT_DIR / f'test_{final_anchor.isoformat()}.parquet'
if REBUILD_SNAPSHOTS or not test_path.exists():
    print(f'Строим финальный v2-срез {final_anchor} ...')
    test_snapshot = build_snapshot(
        data, users, final_anchor, with_target=False, feature_pack=FEATURE_PACK
    )
    save_snapshot(test_snapshot, CATBOOST_V2_SNAPSHOT_DIR, final_anchor, 'test')
else:
    print(f'Используем готовый срез: {test_path.name}')

## 5. Проверка схемы признаков

Все исторические срезы должны иметь одинаковый список и порядок признаков. Отличаться может только наличие `target` в обучающих данных.

In [ ]:
snapshots = load_snapshots(CATBOOST_V2_SNAPSHOT_DIR, kind='train')
test_snapshot = pl.read_parquet(test_path)
feature_cols = feature_columns(snapshots[historical_anchors[0]])

for anchor, snapshot in snapshots.items():
    assert feature_columns(snapshot) == feature_cols, f'Разная схема: {anchor}'
assert feature_columns(test_snapshot) == feature_cols
assert len(feature_cols) == 216

print(f'Исторических срезов: {len(snapshots)}')
print(f'Признаков v2: {len(feature_cols)}')
print(f'Пользователей в финальном срезе: {test_snapshot.height:,}')

## 6. Четыре временных holdout

Используются те же даты и те же правила, что в baseline-backtest. Благодаря этому разность RMSLE отражает изменение признаков, а не изменение состава валидации.

In [ ]:
folds = make_temporal_folds(historical_anchors, VALIDATION_ANCHORS)
for fold in folds:
    print(f'holdout={fold.validation_anchor}: train={list(fold.train_anchors)}')

## 7. Обучение v2 на временных фолдах

На этом этапе параметры CatBoost такие же, как в первой версии. Это позволяет измерить чистый эффект новых признаков. Полное обучение четырёх моделей занимает значительное время.

Если `RUN_MULTIFOLD_VALIDATION=False`, загружаются уже сохранённые результаты сегодняшнего запуска.

In [ ]:
v2_metrics_path = CATBOOST_V2_ARTIFACT_DIR / 'multifold_metrics_default.csv'
v2_oof_path = CATBOOST_V2_ARTIFACT_DIR / 'oof_predictions_default.parquet'

if RUN_MULTIFOLD_VALIDATION:
    v2_metrics, v2_oof = run_temporal_experiment(
        snapshots=snapshots,
        folds=folds,
        features=feature_cols,
        model_factory=make_validation_model,
        keep_oof=True,
        label='catboost_v2_default',
    )
    v2_metrics.to_csv(v2_metrics_path, index=False)
    v2_oof.to_parquet(v2_oof_path, index=False)
    v2_summary = {
        'n_features': len(feature_cols),
        'mean_rmsle': float(v2_metrics['catboost_rmsle'].mean()),
        'global_oof_rmsle': rmsle(
            v2_oof['target'].to_numpy(), v2_oof['prediction'].to_numpy()
        ),
        'std_rmsle': float(v2_metrics['catboost_rmsle'].std(ddof=0)),
    }
    save_json(v2_summary, CATBOOST_V2_ARTIFACT_DIR / 'multifold_summary_default.json')
else:
    if not v2_metrics_path.exists() or not v2_oof_path.exists():
        raise FileNotFoundError(
            'Нет сохранённых результатов v2. Установите RUN_MULTIFOLD_VALIDATION=True.'
        )
    v2_metrics = pd.read_csv(v2_metrics_path)
    v2_oof = pd.read_parquet(v2_oof_path)

display(v2_metrics)

## 8. Сравнение с первым CatBoost

Кандидат принимается, если улучшение повторяется на нескольких датах, а не возникает на одном удачном фолде.

In [ ]:
baseline_metrics = pd.read_csv(MULTIFOLD_VALIDATION_DIR / 'metrics.csv')
comparison = baseline_metrics[[
    'validation_anchor', 'catboost_rmsle'
]].rename(columns={'catboost_rmsle': 'baseline_catboost_rmsle'}).merge(
    v2_metrics[['validation_anchor', 'catboost_rmsle', 'best_iteration']],
    on='validation_anchor',
    how='inner',
).rename(columns={'catboost_rmsle': 'v2_rmsle'})
comparison['v2_improvement'] = (
    comparison['baseline_catboost_rmsle'] - comparison['v2_rmsle']
)

baseline_global = rmsle(
    pd.read_parquet(MULTIFOLD_VALIDATION_DIR / 'oof_predictions.parquet')['target'].to_numpy(),
    pd.read_parquet(MULTIFOLD_VALIDATION_DIR / 'oof_predictions.parquet')['catboost_pred'].to_numpy(),
)
v2_global = rmsle(v2_oof['target'].to_numpy(), v2_oof['prediction'].to_numpy())

display(comparison)
print(f'Baseline global OOF RMSLE: {baseline_global:.6f}')
print(f'V2 global OOF RMSLE:       {v2_global:.6f}')
print(f'Улучшение:                 {baseline_global - v2_global:.6f}')
assert (comparison['v2_improvement'] > 0).all(), 'V2 ухудшилась хотя бы на одном фолде.'

Фактический результат сегодняшнего запуска:

| Holdout | CatBoost baseline | CatBoost v2 | Улучшение |
|---|---:|---:|---:|
| 22.10.2025 | 1.733993 | 1.714851 | 0.019142 |
| 19.11.2025 | 1.772562 | 1.752544 | 0.020018 |
| 17.12.2025 | 1.770604 | 1.752768 | 0.017836 |
| 14.01.2026 | 1.722584 | 1.708149 | 0.014435 |

Глобальный OOF RMSLE снизился с `1.750074` до `1.732202`. Улучшение подтвердилось на всех четырёх датах.

## 9. Проверка посткалибровки

Проверяется линейная коррекция в логарифмическом пространстве:

`calibrated_log_prediction = scale × log_prediction + offset`.

Важно оценить её не только на тех же OOF-прогнозах, на которых параметры подбирались. Поэтому каждый фолд по очереди исключается из подбора калибратора и используется только для проверки.

In [ ]:
def fit_log_calibration(prediction_log: np.ndarray, target_log: np.ndarray) -> tuple[float, float]:
    design = np.column_stack([prediction_log, np.ones_like(prediction_log)])
    scale, offset = np.linalg.lstsq(design, target_log, rcond=None)[0]
    return float(scale), float(offset)

def log_rmse(target_log: np.ndarray, prediction_log: np.ndarray) -> float:
    prediction_log = np.maximum(prediction_log, 0.0)
    return float(np.sqrt(np.mean((target_log - prediction_log) ** 2)))

target_log = np.log1p(v2_oof['target'].to_numpy())
prediction_log = np.log1p(v2_oof['prediction'].to_numpy())
calibration_rows = []

for fold_anchor in sorted(v2_oof['validation_anchor'].unique()):
    calibration_train = v2_oof['validation_anchor'] != fold_anchor
    calibration_valid = ~calibration_train
    scale, offset = fit_log_calibration(
        prediction_log[calibration_train], target_log[calibration_train]
    )
    base_score = log_rmse(
        target_log[calibration_valid], prediction_log[calibration_valid]
    )
    calibrated_score = log_rmse(
        target_log[calibration_valid],
        scale * prediction_log[calibration_valid] + offset,
    )
    calibration_rows.append({
        'validation_anchor': str(fold_anchor),
        'base_rmsle': base_score,
        'calibrated_rmsle': calibrated_score,
        'improvement': base_score - calibrated_score,
        'scale': scale,
        'offset': offset,
    })

calibration_check = pd.DataFrame(calibration_rows)
global_scale, global_offset = fit_log_calibration(prediction_log, target_log)
global_base_score = log_rmse(target_log, prediction_log)
global_calibrated_score = log_rmse(
    target_log, global_scale * prediction_log + global_offset
)
display(calibration_check)
print('Среднее улучшение:', calibration_check['improvement'].mean())
USE_CALIBRATION = bool(
    (calibration_check['improvement'].mean() > 0)
    and (calibration_check['improvement'] > 0).all()
)
print('Применять калибровку:', USE_CALIBRATION)
save_json({
    'scale': global_scale,
    'offset': global_offset,
    'base_oof_rmsle': global_base_score,
    'calibrated_oof_rmsle': global_calibrated_score,
    'leave_one_fold_out': calibration_check.to_dict(orient='records'),
    'accepted': USE_CALIBRATION,
}, CATBOOST_V2_ARTIFACT_DIR / 'calibration.json')

Калибровка была отклонена: на OOF целиком она давала небольшое улучшение, но leave-one-fold-out проверка в среднем показала ухудшение. Поэтому финальные прогнозы v2 не корректируются.

## 10. Финальная модель и submission

Модель обучается на всех восьми размеченных срезах — это 2 миллиона строк. Для текущего submission используется 300 деревьев. Это временная консервативная конфигурация; систематический подбор гиперпараметров будет отдельным следующим ноутбуком.

Если `TRAIN_FINAL_MODEL=False`, ячейка только загружает и проверяет уже созданный CSV.

In [ ]:
if TRAIN_FINAL_MODEL:
    final_model, final_prediction = fit_final_model(
        snapshots=snapshots,
        test_snapshot=test_snapshot,
        features=feature_cols,
        model_factory=lambda: make_final_model(FINAL_ITERATIONS),
    )
    submission = pd.DataFrame({
        'user_id': test_snapshot['user_id'].to_numpy(),
        'predict': np.clip(final_prediction, 0, None),
    })
    submission.to_csv(CATBOOST_V2_SUBMISSION_PATH, index=False, float_format='%.8f')
    final_model.save_model(CATBOOST_V2_ARTIFACT_DIR / 'catboost_v2_300.cbm')
    save_json(feature_cols, CATBOOST_V2_ARTIFACT_DIR / 'feature_columns.json')
    save_json({
        'feature_pack': FEATURE_PACK,
        'n_features': len(feature_cols),
        'n_train_rows': int(sum(snapshot.height for snapshot in snapshots.values())),
        'iterations': FINAL_ITERATIONS,
        'calibration': 'used' if USE_CALIBRATION else 'not_used',
    }, CATBOOST_V2_ARTIFACT_DIR / 'final_config.json')
else:
    if not CATBOOST_V2_SUBMISSION_PATH.exists():
        raise FileNotFoundError(
            'Нет готового submission. Установите TRAIN_FINAL_MODEL=True.'
        )
    submission = pd.read_csv(CATBOOST_V2_SUBMISSION_PATH)

assert list(submission.columns) == ['user_id', 'predict']
assert submission.shape == (250_000, 2)
assert submission['user_id'].is_unique
assert (submission['predict'] >= 0).all()

print(f'Submission: {CATBOOST_V2_SUBMISSION_PATH}')
display(submission.head())
display(submission['predict'].describe(percentiles=[0.5, 0.9, 0.99]))

## Итог эксперимента

Набор `enhanced_v2` принят: он улучшил RMSLE на каждом из четырёх временных holdout. Новый submission сохранён отдельно и не перезаписывает первую модель.

Следующая самостоятельная задача — подбор параметров CatBoost. Она должна быть оформлена отдельным ноутбуком `07_catboost_tuning.ipynb`.